#6. What are the problems you will face when trying to match ORB with SIFT features?

SIFT descriptors are based on histograms of oriented gradients, resulting in a vector of 128 floating-point numbers (float32). These are typically compared using the Euclidean Distance (L2​ norm). In contrast, ORB produces binary descriptors through local intensity tests, resulting in a 256-bit string (interpreted as 32 bytes/dimensions in OpenCV). These are measured using the Hamming Distance, which counts bitwise differences.

This leads to several critical problems:

* Data Type Mismatch: Feature matchers in OpenCV require both sets of descriptors to share the same data type. Attempting to match them directly triggers a type-check error.

* Dimensionality Conflict: SIFT's 128-dimensional vectors cannot be mathematically compared to ORB's 32-byte vectors. Algorithms like Brute-Force or FLANN expect identical vector lengths to calculate distances.

* Semantic Incompatibility: Even if the types and sizes were forced to match, the descriptors represent different physical properties of the image. SIFT captures local gradient distributions, while ORB captures binary patterns of relative intensity. Consequently, any generated "matches" would be semantically meaningless.

In [1]:
import cv2
import time
import matplotlib.pyplot as plt

class FeatureExtractor():
  def __init__(self, feature_type, n_features=1000):
      self.n_features = n_features
      self.extractor = self.create_extractor(feature_type)

  def create_extractor(self, feature_type):
    if feature_type == "SIFT":
      return cv2.SIFT_create(nfeatures=self.n_features)
    elif feature_type == "ORB":
      return cv2.ORB_create(nfeatures=self.n_features)
    elif feature_type == "KAZE":    #KAZE and AKAZE don't have a built-in nfeatures method
      return cv2.KAZE_create()
    elif feature_type == "AKAZE":
      return cv2.AKAZE_create()
    else:
      raise ValueError(f"Unsuported feature type: {feature_type}")

  def extract(self, frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    start_time = time.time()
    keypoints, descriptors = self.extractor.detectAndCompute(gray, None)
    end_time = time.time()

    execution_time = (end_time - start_time) * 1000  # conversion to ms

    return keypoints, descriptors, execution_time


class FeatureMatcher():
    def __init__(self, feature_type):
      # 1. Definir a métrica baseada no tipo (NORM_HAMMING ou NORM_L2)
      self.match_metric = self.get_match_metric(feature_type)
      # 2. Instanciar o cv2.BFMatcher com essa métrica
      self.bf_matcher = cv2.BFMatcher(self.match_metric, crossCheck=False)

    def get_match_metric(self, feature_type):
      if feature_type in ["ORB","AKAZE"]:
        return cv2.NORM_HAMMING
      elif feature_type in ["SIFT","KAZE"]:
        return cv2.NORM_L2
      else:
        raise(ValueError(f"Unsuported feature type: {feature_type}"))

    def match(self, des1, des2, ratio=0.75):
      # 1. Executar knnMatch
      matches = self.bf_matcher.knnMatch(des1,des2,k=2)
      # 2. Filtrar matches: if m.distance < ratio * n.distance: matches.append(m)
      good = []
      for m,n in matches:
        if m.distance < ratio*n.distance:
          good.append([m])
      # 3. Retornar a lista de bons matches
      return good


In [5]:
import cv2
import matplotlib.pyplot as plt

def test_orb_sift_incompatibility():
    img1 = cv2.imread('flu1.png')
    img2 = cv2.imread('flu2.jpg')

    if img1 is None or img2 is None:
        print("No img found.")
        return

    extractor_sift = FeatureExtractor("SIFT", n_features=1000)
    extractor_orb = FeatureExtractor("ORB", n_features=1000)

    kp1, des1, t1 = extractor_sift.extract(img1)
    kp2, des2, t2 = extractor_orb.extract(img2)

    print(f"Shape SIFT: {des1.shape}") # Expected: (N, 128)
    print(f"Shape ORB: {des2.shape}")  # Expected: (N, 32)

    try:
        print("\nTentando realizar o matching...")
        matcher = FeatureMatcher("SIFT")
        good_matches = matcher.match(des1, des2, ratio=0.75)

        img_matches = cv2.drawMatchesKnn(img1, kp1, img2, kp2, good_matches, None, flags=2)
        plt.imshow(cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB))
        plt.show()

    except Exception as e:
        print(f"\nExpected error raised: {e}")
        print("Matcher failed due to different dimentions in descriptors")

test_orb_sift_incompatibility()

Shape SIFT: (215, 128)
Shape ORB: (1000, 32)

Tentando realizar o matching...

Expected error raised: OpenCV(4.13.0) /io/opencv/modules/features2d/src/matchers.cpp:761: error: (-215:Assertion failed) _queryDescriptors.type() == trainDescType in function 'knnMatchImpl'

Matcher failed due to different dimentions in descriptors
